# పాఠం 01 - AI ఏజెంట్లకి పరిచయం

**AI ఏజెంట్లు ప్రారంభికులు కోసం** కోర్సులో మొదటి పాఠాలకు స్వాగతం!

ఒక **AI ఏజెంట్** అనేది తర్కకార్య యంత్రంగా పెద్ద భాషా నమూనాను (LLM) ఉపయోగించే ప్రోగ్రామ్ మరియు యూజర్ తరఫున లక్ష్యాన్ని సాధించడానికి నిజ శ్రేయస్సులో *కార్యాలు* చేయగలదు — APIలను పిలవడం, డేటాబేస్‌లను ప్రశ్నించడం లేదా కోడ్ నడపడం వంటి.

ఈ నోట్బుక్‌లో మీరు మీ మొదటి ఏజెంట్‌ను నిర్మించబడతారు: ఒక **ఈటinerary ఏజెంట్** వేకేషన్ గమ్యస్థానాలను సిఫారసు చేస్తుంది. మార్గంలో మీరు ఎలా నేర్చుకుంటారు:

1. **Microsoft Agent Framework** ఉపయోగించి Microsoft Foundry Agent సర్వీస్‌కి కనెక్ట్ అవ్వడం.
2. ఏజెంట్‌కి ఒక **ఉపకరణం** ఇవ్వడం — అది పిలవగల సాధారణ Python ఫంక్షన్.
3. ఏజెంట్‌ను నడపడం మరియు దాని ప్రతిస్పందనను పరిశీలించడం.
4. ఏజెంట్ యొక్క ప్రతిస్పందనను టోకెన్ ఛైతన్యంతో ప్రసారం చేయడం.


## సెటప్

ఈ నోట్‌బుక్‌ను నడపడానికి ముందు, మీరు అందుకున్నారు అని నిర్ధారించుకోండి:

1. **మైక్రోసాఫ్ట్ Foundry ప్రాజెక్ట్** తో ఒక నికి పంపించబడిన చాట్ మోడల్ (ఉదా., `gpt-4.1-mini`).
2. **Azure CLIలో లాగిన్ అవ్వండి** — మీ టెర్మినల్‌లో `az login` అమలు చేయండి.
3. **అవసరమైన ఎన్విరాన్‌మెంట్ వేరియబుల్స్‌ను సెట్ చేయండి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ మైక్రోసాఫ్ట్ Foundry ప్రాజెక్ట్ ఎండ్పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీరు పంపిణీ చేసిన మోడల్ పేరు.

దిగువ సెల్ మీరు అవసరమైన Python ప్యాకేజీల‌ను ఇన్స్టాల్ చేస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## మీ తొలి ఏజెంట్ సృష్టించడం

ఒక ఏజెంట్‌కు రెండు విషయాలు అవసరం:

- **సూచనలు** ఇది *ఎవరు* అందుకు మరియు *ఎలా ప్రవర్తించాలి* (సిస్టమ్ ప్రాంప్ట్) అనే విషయాలను చెబుతాయి.
- **పరికరాలు** — Python ఫంక్షన్లు `@tool` తో డెకొరేట్ చేయబడి, ఏజెంట్ వాటిని సమాచారం పొందడానికి లేదా చర్యలు తీసుకోవడానికి కాల్ చేయవచ్చు.

క్రింద, ప్రాచీనమైన సెలవుల గమ్యస్థానాల జాబితాను తిరిగి ఇచ్చే ఒక సాధారణ పరికరం నిర్వచించాము. యూజర్ ప్రయాణ సిఫార్సులు అడిగినపుడు ఏజెంట్ ఈ పరికరాన్ని ఉపయోగిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## స్ట్రీమింగ్ స్పందనలు

మరింత ఇంటరాక్టివ్ అనుభవం కోసం మీరు ఏజెంట్ యొక్క స్పందనను **స్ట్రీమ్** చేయవచ్చు. పూర్తి జవాబు కోసం వేచి ఉండకుండానే, ఏజెంట్ సృష్టించబడ్డ టెక్ట్స్ భాగాలను ఇవ్వడం మొదలుపెడుతుంది. ఇది తీస్తే, మీరు నిజ కాలంలో అవుట్‌పుట్‌ను చూపించాలనుకునే చాట్ ఇంటర్ఫేస్‌లలో ఇది చాలా ఉపయోగకరం.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నది ఎలా చేయాలో:

- **Microsoft Foundry Agent Service** కి `FoundryChatClient` ద్వారా కనెక్ట్ అయ్యే ప్రొవైడర్‌ని సృష్టించడం.
- ఏజెంట్ మీ Python ఫంక్షన్లను పిలవగలిగేటట్లుగా `@tool` డెకొరేటర్ ఉపయోగించి టూల్‌ని నిర్వచించడం.
- **యూజర్ సందేశంతో ఏజెంట్‌ని రన్ చేయడం** మరియు దాని ప్రతిస్పందనను ప్రింట్ చేయడం.
- **వాస్తవ కాల అవుట్‌పుట్‌కి ప్రతిస్పందనలను స్ట్రీమ్ చేయడం.**

తదుపరి పాఠంలో మనం ఏజెంటిక్ ఫ్రేమ్‌వర్క్‌లను మరింత లోతుగా పరిశీలించి ఏజెంట్లకు మరింత శక్తివంతమైన టూల్స్ మరియు బహుళ-దశ తర్క సామర్థ్యాలు ఇవ్వడం నేర్చుకుంటాం.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
